# Daily Challenge: Comprehensive Mobile Price Analysis

This notebook explores the **Mobile Price Classification** dataset using **Pandas**, **NumPy**, **SciPy**, and **Matplotlib**.

## Goals
- Load and inspect the dataset
- Clean and preprocess the data
- Compute advanced descriptive statistics
- Test statistical hypotheses across price groups
- Visualize distributions and relationships
- Synthesize insights about the main drivers of mobile price class

## 1. Import libraries

In [ ]:
import os
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats

warnings.filterwarnings("ignore")
plt.style.use("ggplot")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

## 2. Load the dataset

The notebook first tries to load a local `train.csv`.  
If it is not found, it attempts to download the public CSV from the GitHub repository.

In [ ]:
DATA_PATH = Path("train.csv")
RAW_URL = "https://raw.githubusercontent.com/devtlv/DailyChallenge-DataAnalysis-W6D5-Mobile_Price_Classification/main/train.csv"

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    source = str(DATA_PATH)
else:
    df = pd.read_csv(RAW_URL)
    source = RAW_URL

print("Loaded from:", source)
print("Shape:", df.shape)
df.head()

## 3. Initial exploration

In [ ]:
display(df.sample(5, random_state=42))
print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum())

print("\nBasic descriptive statistics:")
display(df.describe())

### Dataset context

This dataset contains mobile phone specifications and a target variable `price_range` with 4 classes.  
The target is ordinal and represents price categories from low to high.

In [ ]:
target_col = "price_range"
feature_cols = [c for c in df.columns if c != target_col]

print("Number of features:", len(feature_cols))
print("Features:")
print(feature_cols)
print("\nTarget classes:", sorted(df[target_col].unique()))

## 4. Data cleaning and preprocessing

The dataset is usually clean, but we still verify missing values and prepare variables for analysis.

In [ ]:
# Check duplicate rows
dup_count = df.duplicated().sum()
print("Duplicate rows:", dup_count)

# If duplicates exist, remove them
if dup_count > 0:
    df = df.drop_duplicates().copy()

# Confirm target is integer-coded
df[target_col] = df[target_col].astype(int)

# Separate numeric and categorical-like features
binary_cols = [c for c in feature_cols if set(df[c].dropna().unique()).issubset({0, 1})]
numeric_cols = [c for c in feature_cols if c not in binary_cols]

print("Binary columns:", binary_cols)
print("Numeric columns:", numeric_cols)

### Optional encoding note

Most columns are already numeric.  
Binary variables such as `blue`, `dual_sim`, `four_g`, `three_g`, `touch_screen`, and `wifi` are already encoded as 0/1, so no extra one-hot encoding is necessary for statistical analysis.

## 5. Feature-wise statistical analysis

For each feature, we compute:
- mean
- median
- mode
- range
- variance
- standard deviation
- skewness
- kurtosis

In [ ]:
def feature_stats(series: pd.Series) -> dict:
    s = series.dropna()
    mode_res = stats.mode(s, keepdims=True)
    mode_value = mode_res.mode[0] if len(mode_res.mode) else np.nan

    return {
        "mean": s.mean(),
        "median": s.median(),
        "mode": mode_value,
        "range": s.max() - s.min(),
        "variance": s.var(ddof=1),
        "std_dev": s.std(ddof=1),
        "skewness": stats.skew(s, bias=False),
        "kurtosis": stats.kurtosis(s, bias=False),
    }

stats_table = pd.DataFrame({col: feature_stats(df[col]) for col in feature_cols}).T
display(stats_table.sort_index())

### Interpretation guide
- **Mean / median** reveal the central tendency of each feature.
- **Range / variance / standard deviation** describe variability.
- **Skewness** indicates asymmetry in the distribution.
- **Kurtosis** highlights tails and outliers.

## 6. Visual exploration with Matplotlib

In [ ]:
# Histograms for selected features
selected_features = ["battery_power", "ram", "px_height", "px_width", "mobile_wt", "price_range"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

for ax, col in zip(axes, selected_features):
    ax.hist(df[col], bins=30)
    ax.set_title(f"Histogram of {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# Box plots by price range for key numerical features
box_features = ["battery_power", "ram", "px_height", "px_width", "mobile_wt", "int_memory"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

for ax, col in zip(axes, box_features):
    data = [df.loc[df[target_col] == cls, col] for cls in sorted(df[target_col].unique())]
    ax.boxplot(data, labels=[str(c) for c in sorted(df[target_col].unique())])
    ax.set_title(f"{col} by price range")
    ax.set_xlabel("price_range")
    ax.set_ylabel(col)

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots: RAM vs price-related variables
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df["ram"], df["price_range"], alpha=0.5)
axes[0].set_title("RAM vs price_range")
axes[0].set_xlabel("ram")
axes[0].set_ylabel("price_range")

axes[1].scatter(df["battery_power"], df["price_range"], alpha=0.5)
axes[1].set_title("Battery power vs price_range")
axes[1].set_xlabel("battery_power")
axes[1].set_ylabel("price_range")

plt.tight_layout()
plt.show()

### Correlation heatmap

In [ ]:
corr = df[feature_cols + [target_col]].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(14, 10))
im = ax.imshow(corr.values, aspect="auto")
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.index)
ax.set_title("Correlation heatmap")

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Correlation")
plt.tight_layout()
plt.show()

## 7. Statistical analysis with NumPy and SciPy

In [ ]:
# Correlation of each feature with the target
target_correlations = {}
for col in feature_cols:
    # Pearson correlation for numeric features and ordinal target
    r, p = stats.pearsonr(df[col], df[target_col])
    target_correlations[col] = {"pearson_r": r, "p_value": p}

corr_target_df = pd.DataFrame(target_correlations).T.sort_values("pearson_r", ascending=False)
display(corr_target_df)

In [ ]:
# Stronger rank-based view for ordinal target
spearman_results = {}
for col in feature_cols:
    r, p = stats.spearmanr(df[col], df[target_col])
    spearman_results[col] = {"spearman_r": r, "p_value": p}

spearman_df = pd.DataFrame(spearman_results).T.sort_values("spearman_r", ascending=False)
display(spearman_df)

In [ ]:
# Correlation between selected features and target as a bar chart
top_corr = corr_target_df["pearson_r"].sort_values(key=lambda s: s.abs(), ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(top_corr.index, top_corr.values)
ax.set_title("Pearson correlation with price_range")
ax.set_xlabel("Feature")
ax.set_ylabel("Correlation")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 8. Hypothesis testing

We compare groups across price ranges using statistical tests.

### 8.1 ANOVA
For each feature, we test whether the mean differs across the four price groups.

In [ ]:
anova_results = []

for col in numeric_cols + binary_cols:
    groups = [df.loc[df[target_col] == cls, col] for cls in sorted(df[target_col].unique())]
    f_stat, p_value = stats.f_oneway(*groups)
    anova_results.append({
        "feature": col,
        "F_statistic": f_stat,
        "p_value": p_value
    })

anova_df = pd.DataFrame(anova_results).sort_values("p_value")
display(anova_df)

### 8.2 Pairwise t-test example

We compare the average `ram` of the lowest price class and the highest price class.

In [ ]:
low_group = df.loc[df[target_col] == df[target_col].min(), "ram"]
high_group = df.loc[df[target_col] == df[target_col].max(), "ram"]

t_stat, p_value = stats.ttest_ind(low_group, high_group, equal_var=False)

print("t-statistic:", t_stat)
print("p-value:", p_value)

alpha = 0.05
if p_value < alpha:
    print("Conclusion: Reject H0. RAM differs significantly between the lowest and highest price classes.")
else:
    print("Conclusion: Fail to reject H0.")

### 8.3 Non-parametric alternative: Kruskal-Wallis test

This is useful when distributions are not close to normal.

In [ ]:
kruskal_results = []

for col in numeric_cols + binary_cols:
    groups = [df.loc[df[target_col] == cls, col] for cls in sorted(df[target_col].unique())]
    h_stat, p_value = stats.kruskal(*groups)
    kruskal_results.append({
        "feature": col,
        "H_statistic": h_stat,
        "p_value": p_value
    })

kruskal_df = pd.DataFrame(kruskal_results).sort_values("p_value")
display(kruskal_df)

## 9. Distribution checks and advanced SciPy functions

In [ ]:
# Normality check on a few important features
for col in ["battery_power", "ram", "px_height", "px_width"]:
    stat_norm, p_norm = stats.normaltest(df[col])
    print(f"{col}: statistic={stat_norm:.4f}, p-value={p_norm:.4g}")

In [ ]:
# Price range groups and feature summaries
group_summary = df.groupby(target_col)[["battery_power", "ram", "px_height", "px_width", "mobile_wt", "int_memory"]].agg(["mean", "median", "std"])
display(group_summary)

In [ ]:
# Point-biserial correlation for binary features against high-price indicator
high_price = (df[target_col] == df[target_col].max()).astype(int)

pb_results = []
for col in binary_cols:
    r, p = stats.pointbiserialr(df[col], high_price)
    pb_results.append({"feature": col, "pointbiserial_r": r, "p_value": p})

pb_df = pd.DataFrame(pb_results).sort_values("pointbiserial_r", ascending=False)
display(pb_df)

## 10. Key insights synthesis

### Likely strongest determinants of mobile price class
- **RAM** is usually the most important predictor.
- **Battery power**, **screen dimensions**, and **camera-related variables** often separate classes.
- **Resolution dimensions** (`px_height`, `px_width`) and **internal memory** also tend to matter.
- Binary features such as Bluetooth or 4G often have weaker direct correlation than performance-related features.

### What the tests suggest
- Many features differ significantly across price classes, especially the performance-oriented ones.
- The relationship between the target and variables like RAM is usually strong and statistically significant.
- Features with weak correlation may still help in combination with others in a predictive model.

## 11. Conclusion

This analysis shows that mobile price categories are primarily driven by **hardware capability** rather than simple connectivity flags.  
The most informative variables are generally those linked to:
- memory and processing power,
- display and resolution,
- battery and camera specifications.

The notebook can be extended with:
- machine learning classification models,
- feature selection,
- model comparison,
- and cross-validation for predictive performance.

## 12. Reflection

### Challenges
- The target variable is categorical but ordered, so choosing the right statistical tests matters.
- Some features are binary, others are continuous, requiring different interpretations.
- Visualizing many features at once can become cluttered.

### Solutions
- Used both **parametric** and **non-parametric** tests.
- Compared feature distributions across price classes.
- Focused visualizations on the most informative variables.
- Summarized results with correlation tables and hypothesis tests.

### Learning outcome
This project strengthened skills in:
- data loading and cleaning,
- descriptive statistics,
- hypothesis testing,
- correlation analysis,
- and clear analytical storytelling.